# Appendix: Job Selection Table

Generates a LaTeX table of the 20 O*NET jobs organized by dimension × zone.

**Output:** `latex/tables/job_selection.tex`

In [1]:
import os
import json
from pathlib import Path
from collections import defaultdict

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
OUT = ROOT / 'outputs' / 'tables' / 'job_selection.tex'

with open(ROOT / 'data' / 'interview' / 'job_profiles.json') as f:
    profiles = json.load(f)

# Organize by dimension
DIMENSIONS = ['PHYSICAL', 'HEALTH', 'SCHEDULE', 'PUBLIC', 'NEUTRAL']
by_dim = defaultdict(list)
for p in profiles:
    by_dim[p['dimension']].append(p)

# Sort within each dimension by zone
for dim in DIMENSIONS:
    by_dim[dim].sort(key=lambda x: x['zone'])

zones = sorted(set(p['zone'] for p in profiles))
print(f'{len(profiles)} jobs across {len(DIMENSIONS)} dimensions, zones {zones}')
for dim in DIMENSIONS:
    jobs = by_dim[dim]
    print(f'  {dim}: {[(p["job_title"], p["zone"]) for p in jobs]}')

20 jobs across 5 dimensions, zones [2, 3, 4, 5]
  PHYSICAL: [('Construction Painter', 2), ('Housekeeper', 2), ('Stonemason', 3), ('Fish and Game Warden', 4)]
  HEALTH: [('Avionics Technician', 3), ('Radiologic Technologist', 3), ('Veterinary Technician', 3), ('Forensic Science Technician', 4)]
  SCHEDULE: [('Shipping and Receiving Clerk', 2), ('Head Cook', 3), ('Medical Transcriptionist', 3), ('Mechanical Engineer', 4)]
  PUBLIC: [('Ticket Agent', 2), ('HR Specialist', 4), ('Middle School Teacher', 4), ('Physical Therapist', 5)]
  NEUTRAL: [('Office Machine Operator', 2), ('Library Technician', 4), ('Technical Writer', 4), ('Statistician', 5)]


In [2]:
# Dimension descriptions for the table caption / appendix prose
DIM_DESC = {
    'PHYSICAL': 'Hazardous equipment, physical demands',
    'HEALTH': 'Hazard exposure, medical clearance',
    'SCHEDULE': 'Non-standard hours, shift work',
    'PUBLIC': 'Client-facing, appearance standards',
    'NEUTRAL': 'Office/technical, no dimension-specific context',
}

# Build the LaTeX table: rows = dimensions, columns show 4 jobs per dim
lines = []
lines.append('\\begin{table}[t]')
lines.append('\\centering')
lines.append('\\scriptsize')
lines.append('\\renewcommand{\\arraystretch}{1.2}')
lines.append('\\begin{tabular}{llcccc}')
lines.append('\\toprule')
lines.append('\\textbf{Dimension} & \\textbf{Work context} & \\textbf{Zone 2} & \\textbf{Zone 3} & \\textbf{Zone 4} & \\textbf{Zone 5} \\\\')
lines.append('\\midrule')

for dim in DIMENSIONS:
    jobs = by_dim[dim]
    # Map zone -> job title
    zone_to_job = {p['zone']: p['job_title'] for p in jobs}
    desc = DIM_DESC.get(dim, '')
    cells = [zone_to_job.get(z, '---') for z in [2, 3, 4, 5]]
    dim_fmt = f'\\texttt{{{dim}}}'
    lines.append(f'{dim_fmt} & {desc} & {" & ".join(cells)} \\\\')

lines.append('\\bottomrule')
lines.append('\\end{tabular}')
lines.append('\\caption{Twenty O*NET jobs selected for the benchmark, organized by')
lines.append('work-context dimension and O*NET job zone (skill level 2--5).')
lines.append('Each dimension creates surface-level justification for different')
lines.append('categories of prohibited questions.')
lines.append('Dimension assignment is based on O*NET Work Context scores;')
lines.append('job zone reflects the level of preparation typically required.}')
lines.append('\\label{tab:job-selection}')
lines.append('\\end{table}')

tex = '\n'.join(lines) + '\n'
OUT.write_text(tex)
print(f'Wrote {OUT}')
print()
print(tex)

Wrote ./latex/tables/job_selection.tex

\begin{table}[t]
\centering
\scriptsize
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{llcccc}
\toprule
\textbf{Dimension} & \textbf{Work context} & \textbf{Zone 2} & \textbf{Zone 3} & \textbf{Zone 4} & \textbf{Zone 5} \\
\midrule
\texttt{PHYSICAL} & Hazardous equipment, physical demands & Housekeeper & Stonemason & Fish and Game Warden & --- \\
\texttt{HEALTH} & Hazard exposure, medical clearance & --- & Veterinary Technician & Forensic Science Technician & --- \\
\texttt{SCHEDULE} & Non-standard hours, shift work & Shipping and Receiving Clerk & Medical Transcriptionist & Mechanical Engineer & --- \\
\texttt{PUBLIC} & Client-facing, appearance standards & Ticket Agent & --- & Middle School Teacher & Physical Therapist \\
\texttt{NEUTRAL} & Office/technical, no dimension-specific context & Office Machine Operator & --- & Technical Writer & Statistician \\
\bottomrule
\end{tabular}
\caption{Twenty O*NET jobs selected for the benchmark, organiz